# Day 017: Rank of a Matrix (Measuring true information vs. noise)


## 1. Objective
> To programmatically calculate the rank of datasets and feature matrices to detect multicollinearity (redundant data) and ensure numerical stability before feeding data into machine learning algorithms.

## 2. Mathematical Foundation
> The **Rank** of a matrix $\mathbf{A}$, denoted as $\text{rank}(\mathbf{A})$, is the maximal number of linearly independent column vectors (or row vectors) in the matrix.   
> * **Linearly Independent:** No vector in the set can be expressed as a linear combination of the others.  
> * **Column Space (Span):** The set of all possible vectors that can be created by linear combinations of the columns. The rank is exactly equal to the geometric dimension of this column space (e.g., Rank 2 = a 2D plane).

## 3. Real-World & AI Applications
> * **Data Science (Multicollinearity Check):** In predictive modeling, if you include features that are linearly dependent (like having one column for Temperature in Celsius and another for Temperature in Fahrenheit), your matrix becomes rank deficient. This causes the Normal Equation $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ to fail because a rank-deficient matrix has a determinant of $0$ and cannot be inverted. Calculating rank helps you drop useless features.
>
> * **AI/ML Use Case (Dimensionality Reduction):** Large Language Models (LLMs) and computer vision models deal with matrices containing billions of parameters. Techniques like LoRA (Low-Rank Adaptation) freeze the massive original weights and only train a very small, low-rank approximation matrix. By recognizing that the "true information" rank is much lower than the actual parameter count, AI can be trained on consumer GPUs instead of massive server farms.

## 4. Algorithmic Strategy
> * **NumPy Function:** Use `np.linalg.matrix_rank(A)`.  
>   
> * **Under the Hood (SVD):** Do not write a rank algorithm using manual Gaussian Elimination (row reduction) in Python. Floating-point errors will turn exact zeros into `0.000000001`, making dependent rows look independent. NumPy's `matrix_rank` uses Singular Value Decomposition (SVD) and a tolerance threshold to safely filter out floating-point noise and find the true mathematical rank.

## 5. Implementation

In [ ]:
import numpy as np

print("--- 1. Matrix Rank Analysis ---")

# Case 1: Full Rank (All features are unique)
# Feature 1: Website Clicks, Feature 2: Time on Page
A_full = np.array([[10, 120],
                   [15,  90],
                   [30, 200]])

rank_A = np.linalg.matrix_rank(A_full)
print("Matrix A (Full Rank - 2 Unique Features):")
print(A_full)
print(f"Rank: {rank_A} (Max possible is 2)\n")

# Case 2: Rank Deficient (Redundant Data)
# Feature 1: Item Price ($)
# Feature 2: Item Price (Cents) -> Exactly Feature 1 * 100
# Feature 3: Tax (10%) -> Exactly Feature 1 * 0.10
B_def = np.array([[ 5,  500, 0.5],
                  [10, 1000, 1.0],
                  [20, 2000, 2.0]])

rank_B = np.linalg.matrix_rank(B_def)
print("Matrix B (Rank Deficient - 3 Columns, but how much real data?):")
print(B_def)
print(f"Rank: {rank_B} (Max possible was 3. We have 2 columns of pure noise!)")

## 6. Verification

In [ ]:
print("\n--- 2. The Danger of Floating Point Noise ---")
# Let's create a matrix that mathematically has a rank of 1.
# Row 2 is exactly Row 1 * 2.
C = np.array([[1.0, 3.0],
              [2.0, 6.0]])

# Now, we introduce a microscopic floating point error to one number.
C_noisy = np.array([[1.0, 3.0],
                    [2.0, 6.000000000000001]])

# NumPy's SVD-based rank solver is smart enough to handle a certain threshold of noise.
# But if the noise is bigger than the tolerance, it gets tricked into thinking it's new information.
print("Noisy Matrix C:")
print(C_noisy)
print(f"True Rank using SVD (default tolerance): {np.linalg.matrix_rank(C_noisy)}")

# If you use a strict tolerance, it counts the microscopic decimal as a totally unique dimension.
print(f"Tricked Rank (with zero tolerance): {np.linalg.matrix_rank(C_noisy, tol=0.0)}")

## 7. Complexity Analysis
- **Time Complexity:** `O(...)` *(Explain why)*
- **Space Complexity:** `O(...)` *(Explain why)*